# Preparation for MULTI-seq demultiplexing for run 'multi_bAllos_4'
Here, the count matrix that is output of cellranger count is loaded and filtered to remove very low-quality cell barcodes. The filtered cell barcode list is then passed to the sample demultiplexing workflow to assign each cell to a sample of origin.  
Note that we use the raw and not the filtered feature matrix here, as we expect that the sample classification pipeline will reliably detect 'real cells' and remove empty droplets. We also filter relatively leniently at first to account for this. 

In [2]:
library(ggrepel)
library(ggplot2)
library(dplyr)
library(Seurat)
library(tidyr) 
library(dplyr)
library(DescTools)

source("../sc_analysis_palettes_and_helpers.r")

# set working directory and sample name variable
outwd <- "data/"

dat_name <- "multi_bAllos_4"

In [3]:
# load raw feature matrix
dat.data <- Read10X(data.dir = "./data/multiseq_bAllos4_PTs_s1/outs/raw_feature_bc_matrix")

# initialise seurat object, only allowing to read in genes that are detected in a minimum of three cells and cells with a minimum of 150 detected genes
dat <- CreateSeuratObject(counts = dat.data, project = "multi_bAllos_4_PTs_S1", min.cells = 3, min.features = 150)

rm(dat.data)


Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”


In [ ]:

# Calculate percenate of reads that map to mitochondrial genome
dat[["percent.mt"]] <- PercentageFeatureSet(object = dat, pattern = "^mt-")

# Plot feature- and UMI-count as well as percentage of mitochondrial reads for all datasets before filtering. Base filtering parameters below on these observations
  
png(paste0(outwd,"pics/filterInf",dat@project.name, ".png"), width = 13, height = 15, units = 'in', res = 300)
    VlnPlot(object = dat, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"), ncol = 3, pt.size = 0.1)
dev.off()

png(paste0(outwd,"pics/nFeature_barplot_",dat@project.name, ".png"), width = 10, height = 8, units = 'in', res = 300)
    hist(dat$nFeature_RNA, breaks = 120, border = "black") # Make plot
dev.off()

png(paste0(outwd,"pics/nFeature_barplot_xlim1000",dat@project.name, ".png"), width = 10, height = 8, units = 'in', res = 300)
    hist(dat$nFeature_RNA, breaks = 500, border = "black", xlim = range(0,1000)) # Make plot
dev.off()

png(paste0(outwd,"pics/nCount_barplot_",dat@project.name, ".png"), width = 10, height = 8, units = 'in', res = 300)
    hist(dat$nCount_RNA, breaks = 120, border = "black") # Make plot
dev.off()

png(paste0(outwd,"pics/nCount_barplot_xlim3000_",dat@project.name, ".png"), width = 10, height = 8, units = 'in', res = 300)
    hist(dat$nCount_RNA, breaks = 4000, border = "black", xlim = range(0,1500)) # Make plot
dev.off()


In [6]:
# Filter
dim(dat)
dat<- subset(x = dat, subset = nFeature_RNA > 200 & nFeature_RNA < 4100 & percent.mt < 10 & nCount_RNA > 330 & nCount_RNA < 50000)
dim(dat)

[1] 22658 41739

[1] 22658 16489

In [8]:
# Save barcodes that remain after filtering to pass to demultiplexing workflow

write.table(colnames(dat), paste0(outwd,"barcodes_after_filter_",dat@project.name,".txt"), sep="\t", quote = FALSE)
